In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [ ]:
#Path Configuration
base_path = r"D:\Karir\Bootcamp\Data Scientist Rakamin\Week 3\Final Task"
train_path = f"{base_path}\\application_train.csv"
test_path  = f"{base_path}\\application_test.csv"

app_train = pd.read_csv(train_path)
app_test  = pd.read_csv(test_path)


In [ ]:
# 1. Bureau & Bureau Balance
bureau = pd.read_csv(f"{base_path}\\bureau.csv")

bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',
    'AMT_CREDIT_SUM': 'sum',
    'AMT_CREDIT_SUM_DEBT': 'sum',
    'AMT_CREDIT_SUM_OVERDUE': 'sum',
    'CREDIT_DAY_OVERDUE': 'max',
    'DAYS_CREDIT': 'mean',
    'AMT_ANNUITY': 'mean'
}).reset_index()

active_loan = bureau[bureau['CREDIT_ACTIVE'] == 'Active'].groupby('SK_ID_CURR').size().reset_index(name='BUREAU_ACTIVE_LOAN_COUNT')
closed_loan = bureau[bureau['CREDIT_ACTIVE'] == 'Closed'].groupby('SK_ID_CURR').size().reset_index(name='BUREAU_CLOSED_LOAN_COUNT')

bureau_agg = bureau_agg.merge(active_loan, on='SK_ID_CURR', how='left')
bureau_agg = bureau_agg.merge(closed_loan, on='SK_ID_CURR', how='left')

bureau_agg.rename(columns={
    'SK_ID_BUREAU': 'BUREAU_LOAN_COUNT',
    'AMT_CREDIT_SUM': 'BUREAU_TOTAL_CREDIT',
    'AMT_CREDIT_SUM_DEBT': 'BUREAU_TOTAL_DEBT',
    'AMT_CREDIT_SUM_OVERDUE': 'BUREAU_TOTAL_OVERDUE',
    'CREDIT_DAY_OVERDUE': 'BUREAU_MAX_OVERDUE_DAYS',
    'DAYS_CREDIT': 'BUREAU_AVG_DAYS_CREDIT',
    'AMT_ANNUITY': 'BUREAU_AVG_ANNUITY'
}, inplace=True)

# Feature Ratios
bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = bureau_agg['BUREAU_TOTAL_DEBT'] / bureau_agg['BUREAU_TOTAL_CREDIT']
bureau_agg['BUREAU_OVERDUE_CREDIT_RATIO'] = bureau_agg['BUREAU_TOTAL_OVERDUE'] / bureau_agg['BUREAU_TOTAL_CREDIT']
bureau_agg['BUREAU_ACTIVE_RATIO'] = bureau_agg['BUREAU_ACTIVE_LOAN_COUNT'] / bureau_agg['BUREAU_LOAN_COUNT']

# Handling inf values
bureau_agg.replace([np.inf, -np.inf], np.nan, inplace=True)

app_train = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')
app_test  = app_test.merge(bureau_agg, on='SK_ID_CURR', how='left')


In [ ]:

# 2. Previous Application
prev = pd.read_csv(f"{base_path}\\previous_application.csv")

prev_agg = prev.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'AMT_APPLICATION': 'mean',
    'AMT_CREDIT': 'mean',
    'AMT_ANNUITY': 'mean',
    'AMT_DOWN_PAYMENT': 'mean',
    'RATE_DOWN_PAYMENT': 'mean',
    'CNT_PAYMENT': 'mean',
    'DAYS_DECISION': 'mean'
}).reset_index()

prev_agg.rename(columns={
    'SK_ID_PREV': 'PREV_APP_COUNT',
    'AMT_APPLICATION': 'PREV_AVG_APPLICATION',
    'AMT_CREDIT': 'PREV_AVG_CREDIT',
    'AMT_ANNUITY': 'PREV_AVG_ANNUITY',
    'AMT_DOWN_PAYMENT': 'PREV_AVG_DOWN_PAYMENT',
    'RATE_DOWN_PAYMENT': 'PREV_AVG_DOWN_PAYMENT_RATE',
    'CNT_PAYMENT': 'PREV_AVG_CNT_PAYMENT',
    'DAYS_DECISION': 'PREV_AVG_DAYS_DECISION'
}, inplace=True)

status_approved = prev[prev['NAME_CONTRACT_STATUS'] == 'Approved'].groupby('SK_ID_CURR').size().reset_index(name='PREV_APPROVED_COUNT')
status_refused = prev[prev['NAME_CONTRACT_STATUS'] == 'Refused'].groupby('SK_ID_CURR').size().reset_index(name='PREV_REFUSED_COUNT')

prev_agg = prev_agg.merge(status_approved, on='SK_ID_CURR', how='left')
prev_agg = prev_agg.merge(status_refused, on='SK_ID_CURR', how='left')
prev_agg['PREV_APPROVAL_RATIO'] = prev_agg['PREV_APPROVED_COUNT'] / prev_agg['PREV_APP_COUNT']
prev_agg.replace([np.inf, -np.inf], np.nan, inplace=True)

app_train = app_train.merge(prev_agg, on='SK_ID_CURR', how='left')
app_test  = app_test.merge(prev_agg, on='SK_ID_CURR', how='left')


In [ ]:
# 3. Installments Payments
inst = pd.read_csv(f"{base_path}\\installments_payments.csv")

inst['PAYMENT_DELAY'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['LATE_PAYMENT'] = (inst['PAYMENT_DELAY'] > 0).astype(int)
inst['PAYMENT_DIFF'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
inst['UNDERPAID'] = (inst['PAYMENT_DIFF'] > 0).astype(int)

inst_agg = inst.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'PAYMENT_DELAY': ['mean', 'max'],
    'LATE_PAYMENT': 'mean',
    'PAYMENT_DIFF': ['mean', 'sum'],
    'UNDERPAID': 'mean',
    'AMT_INSTALMENT': 'sum',
    'AMT_PAYMENT': 'sum'
}).reset_index()

inst_agg.columns = ['_'.join(col).strip('_') for col in inst_agg.columns.values]
inst_agg.rename(columns={
    'SK_ID_PREV_count': 'INSTAL_COUNT',
    'PAYMENT_DELAY_mean': 'INST_AVG_DELAY',
    'PAYMENT_DELAY_max': 'INST_MAX_DELAY',
    'LATE_PAYMENT_mean': 'INST_LATE_RATIO',
    'PAYMENT_DIFF_mean': 'INST_AVG_UNDERPAY',
    'PAYMENT_DIFF_sum': 'INST_TOTAL_UNDERPAY',
    'UNDERPAID_mean': 'INST_UNDERPAID_RATIO',
    'AMT_INSTALMENT_sum': 'INST_TOTAL_INSTAL_AMT',
    'AMT_PAYMENT_sum': 'INST_TOTAL_PAID_AMT'
}, inplace=True)

inst_agg['INST_PAYMENT_RATIO'] = inst_agg['INST_TOTAL_PAID_AMT'] / inst_agg['INST_TOTAL_INSTAL_AMT']
inst_agg.replace([np.inf, -np.inf], np.nan, inplace=True)

app_train = app_train.merge(inst_agg, on='SK_ID_CURR', how='left')
app_test  = app_test.merge(inst_agg, on='SK_ID_CURR', how='left')

In [ ]:

# 4. POS CASH Balance
pos = pd.read_csv(f"{base_path}\\POS_CASH_balance.csv")

pos_agg = pos.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count', 
    'MONTHS_BALANCE': 'max',
    'SK_DPD': ['max', 'mean'],
    'SK_DPD_DEF': ['max', 'mean'],
    'CNT_INSTALMENT_FUTURE': 'mean'
}).reset_index()

pos_agg.columns = ['_'.join(col).strip('_') for col in pos_agg.columns.values]
pos_agg.rename(columns={
    'SK_ID_PREV_count': 'POS_COUNT',
    'SK_DPD_max': 'POS_MAX_DPD',
    'SK_DPD_mean': 'POS_AVG_DPD',
    'CNT_INSTALMENT_FUTURE_mean': 'POS_AVG_FUTURE_INSTALMENTS'
}, inplace=True)

app_train = app_train.merge(pos_agg, on='SK_ID_CURR', how='left')
app_test  = app_test.merge(pos_agg, on='SK_ID_CURR', how='left')


In [ ]:

# 5. Credit Card Balance
cc = pd.read_csv(f"{base_path}\\credit_card_balance.csv")

cc_agg = cc.groupby('SK_ID_CURR').agg({
    'AMT_BALANCE': 'mean',
    'AMT_CREDIT_LIMIT_ACTUAL': 'mean',
    'AMT_DRAWINGS_ATM_CURRENT': 'sum',
    'AMT_DRAWINGS_CURRENT': 'sum',
    'SK_DPD': ['max', 'mean'],
    'CNT_DRAWINGS_ATM_CURRENT': 'mean'
}).reset_index()

cc_agg.columns = ['_'.join(col).strip('_') for col in cc_agg.columns.values]
cc_agg.rename(columns={
    'AMT_DRAWINGS_ATM_CURRENT_sum': 'CC_TOTAL_ATM_DRAWINGS',
    'AMT_BALANCE_mean': 'CC_AVG_BALANCE',
    'SK_DPD_max': 'CC_MAX_DPD'
}, inplace=True)

cc_agg['CC_UTILIZATION_RATIO'] = cc_agg['CC_AVG_BALANCE'] / cc_agg['AMT_CREDIT_LIMIT_ACTUAL_mean']
cc_agg.replace([np.inf, -np.inf], np.nan, inplace=True)

app_train = app_train.merge(cc_agg, on='SK_ID_CURR', how='left')
app_test  = app_test.merge(cc_agg, on='SK_ID_CURR', how='left')


In [ ]:

# 6. Domain Knowledge Features
for dataset in [app_train, app_test]:
    dataset['PAYMENT_RATE'] = dataset['AMT_ANNUITY'] / dataset['AMT_CREDIT']
    dataset['CREDIT_TO_INCOME_RATIO'] = dataset['AMT_CREDIT'] / dataset['AMT_INCOME_TOTAL']
    dataset['ANNUITY_TO_INCOME_RATIO'] = dataset['AMT_ANNUITY'] / dataset['AMT_INCOME_TOTAL']
    dataset['DAYS_EMPLOYED_PERCENT'] = dataset['DAYS_EMPLOYED'] / dataset['DAYS_BIRTH']
    dataset['EXT_SOURCE_MEAN'] = dataset[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
    dataset['EXT_SOURCE_MULT'] = dataset['EXT_SOURCE_1'] * dataset['EXT_SOURCE_2'] * dataset['EXT_SOURCE_3']


In [ ]:
# 7. Output
output_path = f"{base_path}\\Code"
app_train.to_csv(f"{output_path}\\train_full_features.csv", index=False)
app_test.to_csv(f"{output_path}\\test_full_features.csv", index=False)